# 03 — Full Pipeline: Baseline → Dataset → GRPO Training

Flusso completo di sviluppo su **Google Colab** con **Unsloth**:

1. **Setup & Imports** — installa dipendenze, verifica GPU
2. **Genera Dataset** — crea il dataset sintetico (5000 samples)
3. **Baseline Evaluation** — valuta il modello base (senza fine-tuning) e logga su wandb
4. **GRPO Training** — addestra il modello con GRPO e logga su wandb
5. **Post-Training Evaluation** — valuta il modello addestrato e confronta con baseline

> **Requisiti:** GPU su Colab (T4 minimo, A100 ideale)

## ⚙️ Parametri Configurabili

Modifica i valori qui sotto **prima di eseguire** il notebook.
Tutto il resto delle celle legge da queste variabili.
Gli override YAML (impostati a `None`) usano il valore dal file config.

In [ ]:
# ╔════════════════════════════════════════════════════════════╗
# ║  PARAMETRI CONFIGURABILI — modifica qui prima di eseguire  ║
# ║  Tutto il resto del notebook legge da queste variabili.    ║
# ╚════════════════════════════════════════════════════════════╝

# --- Parametri globali ---
WANDB_PROJECT = "grpo-strict-generation"
NUM_SAMPLES = 5000  # campioni nel dataset sintetico
SEED = 42
MAX_EVAL_SAMPLES = 200  # campioni per baseline e post-GRPO eval

# --- Override config YAML (None = usa valore dal YAML) ---
OVERRIDES = {
    # --- Modello ---
    "model.name": None,  # es. "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    "model.quantization": None,  # "4bit", "8bit", None
    "model.use_unsloth": None,  # True/False
    "model.fast_inference": None,  # True/False
    "model.max_seq_length": None,  # es. 2048
    "model.gpu_memory_utilization": None,  # es. 0.75
    "model.num_gpus": None,  # es. 1 (>1 disabilita Unsloth e fast_inference)
    # --- LoRA ---
    "lora.r": None,  # es. 16
    "lora.lora_alpha": None,  # es. 32
    # --- Dataset ---
    "dataset.max_samples": None,  # es. 500 (limita per debug)
    "dataset.thinking": None,  # True/False
    # --- Training ---
    "training.max_steps": None,  # es. 50 per test rapido
    "training.learning_rate": None,  # es. 5e-6
    "training.per_device_train_batch_size": None,
    "training.gradient_accumulation_steps": None,
    "training.logging_steps": None,  # es. 5
    "training.save_steps": None,  # es. 25
    "training.save_total_limit": None,  # es. 2
    # --- GRPO ---
    "grpo.num_generations": None,  # es. 4
    "grpo.max_completion_length": None,  # es. 768
    "grpo.max_prompt_length": None,  # es. 512
    "grpo.beta": None,  # es. 0.04
    "grpo.temperature": None,  # es. 0.9
    # --- Reward ---
    "reward.weight_format": None,  # es. 0.20
    "reward.weight_validity": None,  # es. 0.35
    "reward.weight_schema": None,  # es. 0.35
    "reward.weight_reasoning": None,  # es. 0.10
}


def apply_overrides(config: dict, overrides: dict) -> dict:
    """Applica gli override non-None al config dict (modifica in-place)."""
    applied = []
    for dotted_key, value in overrides.items():
        if value is None:
            continue
        keys = dotted_key.split(".")
        d = config
        for k in keys[:-1]:
            d = d.setdefault(k, {})
        old = d.get(keys[-1], "<missing>")
        d[keys[-1]] = value
        applied.append(f"  {dotted_key}: {old} → {value}")
    if applied:
        print(f"[overrides] {len(applied)} parametri sovrascritti:")
        for line in applied:
            print(line)
    else:
        print("[overrides] Nessun override — si usano i valori dal YAML")

    # Auto-disable Unsloth e fast_inference se num_gpus > 1
    num_gpus = config.get("model", {}).get("num_gpus", 1)
    if num_gpus > 1:
        if config["model"].get("use_unsloth", False):
            config["model"]["use_unsloth"] = False
            print(
                f"[overrides] num_gpus={num_gpus} → use_unsloth forzato a False"
            )
        if config["model"].get("fast_inference", False):
            config["model"]["fast_inference"] = False
            print(
                f"[overrides] num_gpus={num_gpus} → fast_inference forzato a False"
            )

    return config


print("Parametri configurabili pronti.")

## 1. Setup su Colab

In [ ]:
!rm -rf GRPO-strict-generation
!git clone https://github.com/GiuseppeBellamacina/GRPO-strict-generation.git
%cd GRPO-strict-generation
!bash setup.sh

## 2. Imports & GPU Check

In [ ]:
import os

# vLLM standby mode disabilitata (0): su Colab il runtime forza
# expandable_segments:True nell'allocator CUDA prima dell'avvio del kernel,
# incompatibile con standby mode. Su A100 o runtime custom si può provare "1".
os.environ["UNSLOTH_VLLM_STANDBY"] = "0"

In [ ]:
from unsloth import FastLanguageModel

print("Unsloth importato correttamente")

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
    )
else:
    print("WARNING: Nessuna GPU trovata!")

import transformers
import trl

print(f"transformers: {transformers.__version__}")
print(f"trl:          {trl.__version__}")

In [ ]:
import wandb

# Login a wandb — la prima volta su Colab chiede l'API key.
# Prendi la tua key da https://wandb.ai/authorize
wandb.login()

In [ ]:
# Entra nella repo se il clone è appena avvenuto e cwd è ancora /content
import os
from pathlib import Path

_tris = Path.cwd() / "tris"
if _tris.is_dir() and (_tris / "pyproject.toml").exists():
    os.chdir(_tris)
    print(f"cd → {_tris}")
else:
    print(f"cwd già corretto: {Path.cwd()}")

In [ ]:
import os
from pathlib import Path


# Resolve repo root robustly:
# - normal: cwd is already /content/tris → pyproject.toml found immediately
# - restarted kernel: cwd is /content → Colab fallback kicks in
def _find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    # Colab-specific fallback: repo always cloned to /content/tris
    colab_path = Path("/content/tris")
    if colab_path.exists():
        return colab_path
    raise RuntimeError(
        "Repo root non trovato. Esegui prima la cella di setup (git clone)."
    )


ROOT = _find_repo_root()
os.chdir(ROOT)

# Costanti (WANDB_PROJECT definito nella cella parametri in alto)
DATASET_PATH = ROOT / "data" / "synthetic"
CONFIG_PATH = ROOT / "experiments" / "configs" / "grpo.yaml"

# Esporta il progetto wandb come env var così il Trainer di HuggingFace
# non cade sul default "huggingface" se deve auto-inizializzare wandb.
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

print(f"Root progetto: {ROOT}")

## 3. Genera Dataset Sintetico

Generiamo un dataset sintetico con `NUM_SAMPLES` samples (3 livelli di difficoltà: simple, medium, hard).

In [ ]:
from src.datasets.dataloader import load_synthetic_dataset
from src.datasets.synthetic_dataset import generate_dataset

# NUM_SAMPLES e SEED definiti nella cella parametri in alto

if DATASET_PATH.exists():
    print(f"Dataset già presente in {DATASET_PATH}, lo ricarico...")
else:
    print(f"Genero {NUM_SAMPLES} samples (seed={SEED})...")
    ds = generate_dataset(num_samples=NUM_SAMPLES, seed=SEED)
    ds.save_to_disk(str(DATASET_PATH))
    print(f"Dataset salvato in {DATASET_PATH}")

ds = load_synthetic_dataset(str(DATASET_PATH))

for split_name, split_ds in ds.items():
    print(f"  {split_name}: {len(split_ds)} samples")
    # Distribuzione per difficoltà
    diffs = split_ds["difficulty"]
    for d in ["simple", "medium", "hard"]:
        count = sum(1 for x in diffs if x == d)
        print(f"    json/{d}: {count}")

## 4. Baseline Evaluation

Valutiamo il modello **base** (senza fine-tuning) sul test set.
Questo ci dà il punto di partenza per misurare il miglioramento dopo GRPO.

In [ ]:
from src.datasets.dataloader import format_prompt_for_model
from src.evaluation.baseline_eval import generate_completions
from src.evaluation.evaluate import compute_detailed_metrics
from src.models.model_loader import load_model_and_tokenizer
from src.utils.config import load_config

config = load_config(str(CONFIG_PATH))
config = apply_overrides(config, OVERRIDES)

# Carica modello BASE (senza LoRA, senza fast_inference):
# - no "lora" key → no adapters
# - fast_inference=False → no vLLM, no CUDA graph init
# Questo evita il crash del CUDA allocator (use_count > 0) quando
# il modello di training carica vLLM nello stesso processo subito dopo.
base_config = {"model": {**config["model"], "fast_inference": False}}
print(f"Caricamento modello base: {config['model']['name']}")
model, tokenizer = load_model_and_tokenizer(base_config)

In [ ]:
# Prepara prompt dal test set (MAX_EVAL_SAMPLES definito nella cella parametri in alto)
test_ds = ds["test"]
eval_ds = test_ds.select(range(min(MAX_EVAL_SAMPLES, len(test_ds))))

prompts = [
    format_prompt_for_model(eval_ds[i], tokenizer) for i in range(len(eval_ds))
]
difficulties = list(eval_ds["difficulty"])

print(f"Prompts preparati: {len(prompts)}")

# Genera completions
gen_config = {
    "max_new_tokens": 512,
    "temperature": 0.7,
    "top_p": 0.95,
    "do_sample": True,
}

print("Generazione completions baseline...")
baseline_completions = generate_completions(
    model=model,
    tokenizer=tokenizer,
    prompts=prompts,
    generation_config=gen_config,
    num_return_sequences=1,
    batch_size=4,
)
print(f"Completions generate: {len(baseline_completions)}")

In [ ]:
import json

# Calcola metriche baseline
first_completions = [comps[0] for comps in baseline_completions]
baseline_metrics = compute_detailed_metrics(first_completions, difficulties)

print(f"\n{'='*50}")
print(f"BASELINE — {config['model']['name']}")
print(f"{'='*50}")
print(f"Overall Pass@1: {baseline_metrics['overall_pass_rate']:.4f}")
print("\nPer-category breakdown:")
for cat, stats in baseline_metrics["per_category"].items():
    print(
        f"  {cat}: {stats['pass_rate']:.4f} ({stats['valid']}/{stats['total']})"
    )

# Salva risultati baseline in JSON
baseline_output_dir = ROOT / "experiments" / "checkpoints" / "baseline"
baseline_output_dir.mkdir(parents=True, exist_ok=True)

baseline_json = {
    "model": config["model"]["name"],
    "overall_pass_rate": baseline_metrics["overall_pass_rate"],
    "per_category": baseline_metrics["per_category"],
}
baseline_json_path = baseline_output_dir / "baseline_results.json"
baseline_json_path.write_text(
    json.dumps(baseline_json, indent=2, default=str), encoding="utf-8"
)
print(f"\nBaseline results saved to {baseline_json_path}")

# Log su wandb
wandb.init(
    project=WANDB_PROJECT,
    name=f"baseline-{config['model']['name'].split('/')[-1]}",
    tags=["baseline", "notebook-03"],
    config={"model": config["model"], "eval_samples": len(prompts)},
)

wandb_metrics = {"overall_pass_rate": baseline_metrics["overall_pass_rate"]}
for cat, stats in baseline_metrics["per_category"].items():
    wandb_metrics[f"pass_rate/{cat}"] = stats["pass_rate"]

# Bar chart + scalari per wandb
table_data = [[k, v] for k, v in wandb_metrics.items()]
bar_table = wandb.Table(data=table_data, columns=["metric", "value"])
wandb.log(
    {
        "eval/pass_rates": wandb.plot.bar(
            bar_table, "metric", "value", title="Baseline Pass Rates"
        ),
        **wandb_metrics,
    }
)

wandb.finish()
print("\nBaseline loggata su wandb")

In [ ]:
# Qualche esempio di completions baseline
from src.training.rewards import combined_reward

print("Esempi di completions baseline:\n")
for i in [0, 1, 5, 10]:
    if i >= len(eval_ds):
        break
    comp = first_completions[i]
    r = combined_reward(comp)
    print(f"--- [{difficulties[i]}] reward={r} ---")
    print(f"Prompt:     {eval_ds[i]['prompt'][:100]}...")
    print(f"Completion: {comp[:200]}")
    print()

In [ ]:
# Libera VRAM dal modello base
del model
torch.cuda.empty_cache()
print("VRAM liberata")

## 5. GRPO Training

Addestriamo il modello con **GRPO** (Group Relative Policy Optimization).
Il config viene letto da `experiments/configs/grpo.yaml`.

Parametri chiave:
- **learning_rate**: 5e-6
- **optim**: paged_adamw_8bit (meno VRAM)
- **max_grad_norm**: 0.1 (gradient clipping aggressivo)
- **num_generations**: 8 (gruppo per calcolare i vantaggi relativi)
- **beta**: 0.04 (KL penalty)

> **fast_inference**: se `true` nel config, usa vLLM interno per generazione 2-5x più veloce.
> Si attiva automaticamente solo su Linux con vLLM installato; altrimenti fallback a `model.generate()`.

> **UNSLOTH_VLLM_STANDBY**: disabilitato su Colab T4 perché il runtime forza
> `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` prima dell'avvio del kernel,
> incompatibile con la standby mode. Su A100 o runtime custom potrebbe funzionare.


In [ ]:
from datasets import Dataset
from src.training.rewards import build_reward_function

# Ricarica modello CON LoRA per il training
print(f"Caricamento modello con LoRA: {config['model']['name']}")
model, tokenizer = load_model_and_tokenizer(config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parametri totali:    {total:,}")
print(f"Parametri trainable: {trainable:,} ({100 * trainable / total:.2f}%)")

In [ ]:
# Prepara dataset di training (prompt formattati)
train_ds = ds["train"]
formatted = []
for i in range(len(train_ds)):
    s = train_ds[i]
    p = format_prompt_for_model(s, tokenizer)
    formatted.append({"prompt": p})

prompt_dataset = Dataset.from_list(formatted)
print(f"Training dataset: {len(prompt_dataset)} prompt pronti")

# Reward function — passa thinking dal config così i pesi si adattano automaticamente
thinking = config.get("dataset", {}).get("thinking", True)
reward_fn = build_reward_function(config["reward"], thinking=thinking)
print(f"Reward function creata (thinking={'on' if thinking else 'off'})")

In [ ]:
import datetime

from trl import GRPOConfig

training_cfg = config["training"]
grpo_cfg = config["grpo"]

output_dir = str(ROOT / "experiments" / "checkpoints" / "grpo_full")
log_dir = str(ROOT / "experiments" / "logs" / "grpo_full")
Path(output_dir).mkdir(parents=True, exist_ok=True)
Path(log_dir).mkdir(parents=True, exist_ok=True)

_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"grpo-full-{_ts}"

# bf16 vs fp16 in base alla GPU
_supports_bf16 = (
    torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
)
use_bf16 = _supports_bf16
use_fp16 = not _supports_bf16 and torch.cuda.is_available()

grpo_config = GRPOConfig(
    output_dir=output_dir,
    run_name=run_name,
    # Training
    max_steps=training_cfg.get("max_steps", 1000),
    per_device_train_batch_size=training_cfg.get(
        "per_device_train_batch_size", 1
    ),
    gradient_accumulation_steps=training_cfg.get(
        "gradient_accumulation_steps", 8
    ),
    learning_rate=training_cfg.get("learning_rate", 5e-6),
    lr_scheduler_type=training_cfg.get("lr_scheduler_type", "cosine"),
    warmup_ratio=training_cfg.get("warmup_ratio", 0.1),
    optim=training_cfg.get("optim", "paged_adamw_8bit"),
    weight_decay=training_cfg.get("weight_decay", 0.1),
    max_grad_norm=training_cfg.get("max_grad_norm", 0.1),
    bf16=use_bf16,
    fp16=use_fp16,
    # GRPO
    num_generations=grpo_cfg.get("num_generations", 8),
    max_completion_length=grpo_cfg.get("max_completion_length", 512),
    max_prompt_length=grpo_cfg.get("max_prompt_length", 256),
    beta=grpo_cfg.get("beta", 0.04),
    temperature=grpo_cfg.get("temperature", 0.7),
    # Logging
    logging_dir=log_dir,
    logging_steps=training_cfg.get("logging_steps", 10),
    save_steps=training_cfg.get("save_steps", 200),
    save_total_limit=training_cfg.get("save_total_limit", 3),
    report_to="wandb",
)

# Inizializza il run wandb PRIMA del trainer, così il GRPOTrainer
# riusa il run attivo anziché crearne uno con project="huggingface".
wandb.init(
    project=WANDB_PROJECT,
    name=run_name,
    tags=["grpo", "notebook-03", config["model"]["name"].split("/")[-1]],
    config=config,
)

print("GRPOConfig creata")
print(f"  run_name:       {grpo_config.run_name}")
print(f"  max_steps:      {grpo_config.max_steps}")
print(f"  num_generations: {grpo_config.num_generations}")
print(f"  batch_size:     {grpo_config.per_device_train_batch_size}")
print(f"  grad_accum:     {grpo_config.gradient_accumulation_steps}")
print(f"  learning_rate:  {grpo_config.learning_rate}")
print(f"  optim:          {grpo_config.optim}")
print(f"  beta:           {grpo_config.beta}")
print(f"  bf16={use_bf16}, fp16={use_fp16}")

In [ ]:
from trl import GRPOTrainer

from src.training.callbacks import HighPrecisionLogCallback, WandbAlertCallback

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=prompt_dataset,
    reward_funcs=reward_fn,
    processing_class=tokenizer,
    callbacks=[HighPrecisionLogCallback(), WandbAlertCallback()],
)

print(f"Trainer inizializzato (tipo: {type(trainer).__name__})")
print(f"\nAvvio GRPO training ({grpo_config.max_steps} step)...\n")

trainer.train()
print("\nTraining completato!")

In [ ]:
# Salva il modello finale
final_path = Path(output_dir) / "final"
trainer.save_model(str(final_path))
tokenizer.save_pretrained(str(final_path))
print(f"Modello salvato in {final_path}")

# Chiudi il run wandb del training prima di aprirne uno nuovo per l'eval
wandb.finish()
print("Run wandb training chiuso")

## 6. Post-Training Evaluation

Valutiamo il modello addestrato sullo **stesso test set** usato per la baseline.

In [ ]:
# Genera completions con il modello trainato
# Usiamo gli stessi prompt della baseline per un confronto diretto
print("Generazione completions post-GRPO...")
grpo_completions = generate_completions(
    model=model,
    tokenizer=tokenizer,
    prompts=prompts,
    generation_config=gen_config,
    num_return_sequences=1,
    batch_size=4,
)

# Calcola metriche post-GRPO
grpo_first = [comps[0] for comps in grpo_completions]
grpo_metrics = compute_detailed_metrics(grpo_first, difficulties)

print(f"\n{'='*50}")
print(f"POST-GRPO — {config['model']['name']}")
print(f"{'='*50}")
print(f"Overall Pass@1: {grpo_metrics['overall_pass_rate']:.4f}")
print("\nPer-category breakdown:")
for cat, stats in grpo_metrics["per_category"].items():
    print(
        f"  {cat}: {stats['pass_rate']:.4f} ({stats['valid']}/{stats['total']})"
    )

# Salva risultati post-GRPO in JSON
grpo_json = {
    "model": config["model"]["name"],
    "overall_pass_rate": grpo_metrics["overall_pass_rate"],
    "per_category": grpo_metrics["per_category"],
}
grpo_json_path = Path(output_dir) / "grpo_eval_results.json"
grpo_json_path.write_text(
    json.dumps(grpo_json, indent=2, default=str), encoding="utf-8"
)
print(f"\nGRPO eval results saved to {grpo_json_path}")

# Log su wandb
wandb.init(
    project=WANDB_PROJECT,
    name=f"grpo-eval-{config['model']['name'].split('/')[-1]}",
    tags=["grpo-eval", "notebook-03"],
    config=config,
)

grpo_wandb = {"overall_pass_rate": grpo_metrics["overall_pass_rate"]}
for cat, stats in grpo_metrics["per_category"].items():
    grpo_wandb[f"pass_rate/{cat}"] = stats["pass_rate"]
wandb.log(grpo_wandb)

wandb.finish()
print("\nMetriche post-GRPO loggate su wandb")

## 7. Confronto Baseline vs GRPO

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

all_cats = sorted(
    set(
        list(baseline_metrics["per_category"].keys())
        + list(grpo_metrics["per_category"].keys())
    )
)
labels = ["overall"] + all_cats
b_values = [baseline_metrics["overall_pass_rate"]] + [
    baseline_metrics["per_category"].get(c, {}).get("pass_rate", 0.0)
    for c in all_cats
]
g_values = [grpo_metrics["overall_pass_rate"]] + [
    grpo_metrics["per_category"].get(c, {}).get("pass_rate", 0.0)
    for c in all_cats
]
deltas = [g - b for g, b in zip(g_values, b_values)]

x = np.arange(len(labels))
width = 0.32

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"Baseline vs Post-GRPO — {config['model']['name']}",
    fontsize=13,
    fontweight="bold",
)

# ── Grouped bar: pass rate per categoria ──────────────────────────────────────
ax = axes[0]
bars_b = ax.bar(
    x - width / 2,
    b_values,
    width,
    label="Baseline",
    color="#4C72B0",
    alpha=0.85,
)
bars_g = ax.bar(
    x + width / 2,
    g_values,
    width,
    label="Post-GRPO",
    color="#DD8452",
    alpha=0.85,
)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Pass@1")
ax.set_title("Pass Rate per Category")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.legend()
ax.bar_label(bars_b, fmt="%.2f", padding=3, fontsize=8)
ax.bar_label(bars_g, fmt="%.2f", padding=3, fontsize=8)
ax.axhline(0, color="black", linewidth=0.5)

# ── Delta bar: miglioramento per categoria ────────────────────────────────────
ax2 = axes[1]
colors = ["#2ca02c" if d >= 0 else "#d62728" for d in deltas]
bars_d = ax2.bar(x, deltas, width * 1.8, color=colors, alpha=0.85)
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.set_ylabel("Δ Pass@1 (GRPO − Baseline)")
ax2.set_title("Delta per Category")
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=20, ha="right")
ax2.bar_label(bars_d, fmt="%+.3f", padding=3, fontsize=8)

fig.tight_layout()

# ── Salva figura ──────────────────────────────────────────────────────────────
figures_dir = Path("../figures")
figures_dir.mkdir(parents=True, exist_ok=True)
fig_path = figures_dir / "baseline_vs_grpo_comparison.png"
fig.savefig(fig_path, dpi=150)
print(f"Figure saved to {fig_path}")

plt.show()

# Tabella testuale di riepilogo
print(f"\n{'Categoria':<30} {'Baseline':>10} {'Post-GRPO':>10} {'Delta':>10}")
print("-" * 62)
for lbl, b, g, d in zip(labels, b_values, g_values, deltas):
    print(f"  {lbl:<28} {b:>10.4f} {g:>10.4f} {d:>+10.4f}")
delta_overall = g_values[0] - b_values[0]
print()
if delta_overall > 0:
    print(
        f"GRPO ha migliorato il pass rate di {delta_overall:+.4f} ({delta_overall / max(b_values[0], 1e-6) * 100:+.1f}%)"
    )
elif delta_overall < 0:
    print(f"GRPO ha peggiorato il pass rate di {delta_overall:.4f}")
else:
    print("Nessun cambiamento")

# Salva anche un JSON di confronto
comparison_json = {
    "model": config["model"]["name"],
    "labels": labels,
    "baseline": b_values,
    "grpo": g_values,
    "delta": deltas,
}
comparison_json_path = Path(output_dir) / "comparison_results.json"
comparison_json_path.write_text(
    json.dumps(comparison_json, indent=2, default=str), encoding="utf-8"
)
print(f"Comparison results saved to {comparison_json_path}")

In [ ]:
# Qualche esempio di completions post-GRPO
print("Esempi di completions post-GRPO:\n")
for i in [0, 1, 5, 10]:
    if i >= len(eval_ds):
        break
    comp = grpo_first[i]
    r = combined_reward(comp)
    print(f"--- [{difficulties[i]}] reward={r} ---")
    print(f"Prompt:     {eval_ds[i]['prompt'][:100]}...")
    print(f"Completion: {comp[:200]}")
    print()

In [ ]:
# Pulizia VRAM
del trainer
del model
torch.cuda.empty_cache()
print("VRAM liberata")

## Riepilogo

Questo notebook ha eseguito il flusso completo:

1. **Dataset** — `NUM_SAMPLES` samples JSON (simple/medium/hard)
2. **Baseline** — valutazione modello base → metriche su wandb
3. **GRPO Training** — addestramento con reward rule-based → loss/reward su wandb
4. **Post-GRPO** — valutazione modello trainato → confronto con baseline

Tutti i run sono su **wandb** sotto il progetto `WANDB_PROJECT`.

**Comandi CLI equivalenti:**
```bash
# Baseline
uv run python -m src.evaluation.baseline_eval --config experiments/configs/baseline.yaml

# GRPO Training
uv run python -m src.training.grpo_train --config experiments/configs/grpo.yaml
```